# Clean.ipynb

## Cell 1 — Config
- Set `DATA_DIR` (one variable to change per machine)
- Output goes to `DATA_DIR/meta/`

## Cell 2 — Deduplicate STimage against HEST
- **Inputs**
  - `STimage-1K4M/meta/meta_all_gene.csv` — 1,149 slides
  - `HEST/assets/HEST_v1_1_0.csv` — all HEST samples
- **Build HEST lookup**
  - Key: `dataset_title` (lowercased)
  - Value: list of `subseries` labels for that study
- **For each STimage slide → 3-way classification**
  - Extract study title(s) from the `title` field
    - Handles multi-paper cells: `"Title 1: Foo. Title 2: Bar."`
  - **No title match** → `cleaned` (946) — not in HEST, safe to convert
  - **Title matches + subseries is substring of slide name** → `duplicates` (123) — confirmed in HEST, silently dropped
  - **Title matches but subseries not found in slide name** → `ambiguous` (80) — same study, different naming convention, needs human review
- **Outputs**
  - `cleaned_metadata.csv` — 946 rows, same columns as input
  - `ambiguous_metadata.csv` — 80 rows + two extra debug columns:
    - `_matched_hest_title` — which HEST study matched
    - `_hest_subseries` — HEST's subseries labels for that study

## Cell 3 — Manual review gate
- Tells user to open `ambiguous_metadata.csv` and delete confirmed duplicates
- Prompts `[Y/N]`
  - **Y** — appends surviving ambiguous rows into `cleaned_metadata.csv`
  - **N** — leaves `cleaned_metadata.csv` unchanged
- Final `cleaned_metadata.csv` = the definitive list of STimage slides to convert

In [1]:
import csv
import re
import os

# ── Paths ──────────────────────────────────────────────────────────────────
STIMAGE_META   = "/Users/bradzap/Developer/GitHub/STimage-1K4M/meta/meta_all_gene.csv"
HEST_META      = "/Users/bradzap/Developer/GitHub/HEST/assets/HEST_v1_1_0.csv"
OUTPUT_DIR     = "/Users/bradzap/Developer/meta"

# ── Load data ──────────────────────────────────────────────────────────────
with open(HEST_META, encoding="utf-8-sig") as f:
    hest = list(csv.DictReader(f))

with open(STIMAGE_META) as f:
    stimage = list(csv.DictReader(f))

# ── Build HEST lookup: normalised title → list of subseries ───────────────
# One paper can have many slides in HEST; collect all their subseries labels.
hest_title_to_subseries: dict[str, list[str]] = {}
for row in hest:
    title = row["dataset_title"].strip().lower()
    sub   = row["subseries"].strip()
    hest_title_to_subseries.setdefault(title, [])
    if sub:
        hest_title_to_subseries[title].append(sub)

# ── Helper: extract individual titles from an STimage title cell ───────────
# STimage sometimes encodes multiple papers: "Title 1: Foo. Title 2: Bar."
def extract_titles(raw: str) -> list[str]:
    titles = re.findall(r"Title \d+: (.+?)(?= Title \d+:|$)", raw)
    return [t.strip() for t in titles] if titles else [raw.strip()]

# ── Classify each STimage slide ────────────────────────────────────────────
cleaned    = []   # no title match in HEST → safe to convert
ambiguous  = []   # title match but subseries not found in slide name → manual check
duplicates = []   # title match AND subseries substring confirmed → skip

for row in stimage:
    slide = row["slide"]
    titles = extract_titles(row["title"])

    matched_title = None
    for t in titles:
        if t.lower() in hest_title_to_subseries:
            matched_title = t.lower()
            break

    if matched_title is None:
        cleaned.append(row)
        continue

    # Study is in HEST — check if this specific slide's ID appears in HEST subseries
    subseries_list = hest_title_to_subseries[matched_title]
    confirmed = any(sub and sub in slide for sub in subseries_list)

    if confirmed:
        duplicates.append(row)
    else:
        # Same study but slide label didn't string-match any HEST subseries
        row["_matched_hest_title"] = matched_title
        row["_hest_subseries"]     = " | ".join(subseries_list[:10])
        ambiguous.append(row)

# ── Write outputs ──────────────────────────────────────────────────────────
fieldnames = list(stimage[0].keys())
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "cleaned_metadata.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader(); w.writerows(cleaned)

amb_fields = fieldnames + ["_matched_hest_title", "_hest_subseries"]
with open(os.path.join(OUTPUT_DIR, "ambiguous_metadata.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=amb_fields, extrasaction="ignore")
    w.writeheader(); w.writerows(ambiguous)

# ── Summary ────────────────────────────────────────────────────────────────
print(f"Total STimage slides : {len(stimage)}")
print(f"Confirmed duplicates : {len(duplicates)}  (skipped)")
print(f"Ambiguous            : {len(ambiguous)}   → ambiguous_metadata.csv")
print(f"Safe to convert      : {len(cleaned)}  → cleaned_metadata.csv")

Total STimage slides : 1149
Confirmed duplicates : 123  (skipped)
Ambiguous            : 80   → ambiguous_metadata.csv
Safe to convert      : 946  → cleaned_metadata.csv


In [2]:
import csv
import os

CLEANED_PATH   = os.path.join(OUTPUT_DIR, "cleaned_metadata.csv")
AMBIGUOUS_PATH = os.path.join(OUTPUT_DIR, "ambiguous_metadata.csv")

print("Ambiguous slides are in:")
print(f"  {AMBIGUOUS_PATH}")
print()
print("Please open it, review each slide, and delete any rows that are")
print("confirmed duplicates of HEST samples. Save the file when done.")
print()
answer = input("Merge remaining ambiguous rows into cleaned_metadata.csv? [Y/N]: ").strip().upper()

if answer == "Y":
    with open(AMBIGUOUS_PATH, newline="") as f:
        amb_rows = list(csv.DictReader(f))

    # Strip the extra columns added during classification before appending
    with open(CLEANED_PATH, newline="") as f:
        cleaned_fields = csv.DictReader(f).fieldnames

    with open(CLEANED_PATH, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cleaned_fields, extrasaction="ignore")
        w.writerows(amb_rows)

    print(f"Appended {len(amb_rows)} rows to cleaned_metadata.csv.")
    print(f"Total safe-to-convert slides: {sum(1 for _ in open(CLEANED_PATH)) - 1}")

elif answer == "N":
    print("Ambiguous rows not added. cleaned_metadata.csv unchanged.")
else:
    print("Unrecognised input — nothing written. Re-run this cell and enter Y or N.")

Ambiguous slides are in:
  /Users/bradzap/Developer/meta/ambiguous_metadata.csv

Please open it, review each slide, and delete any rows that are
confirmed duplicates of HEST samples. Save the file when done.

Ambiguous rows not added. cleaned_metadata.csv unchanged.
